In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.functions import col, count, avg, sum as spark_sum, min as spark_min, max as spark_max, year, month, datediff, when, expr, round as spark_round
from pyspark.sql.window import Window

## 1. Temporal Analysis - Yearly CVE Counts

In [0]:
SELECT 
    YEAR(date_published) as year,
    MONTH(date_published) as month,
    WEEKOFYEAR(date_published) as week,
    COUNT(*) as cve_count
FROM cve_silver.core
GROUP BY YEAR(date_published), MONTH(date_published), WEEKOFYEAR(date_published)
ORDER BY year, month, week

year,month,week,cve_count
2024,1,1,92
2024,1,2,273
2024,1,3,270
2024,1,4,310
2024,1,5,189
2024,2,5,125
2024,2,6,422
2024,2,7,442
2024,2,8,443
2024,2,9,337


Databricks visualization. Run in Databricks to view.

## 2. Publication Latency Analysis 

In [0]:
SELECT 
    ROUND(AVG(DATEDIFF(date_published, date_reserved)), 1) as avg_days_to_publish,
    MIN(DATEDIFF(date_published, date_reserved)) as min_days,
    MAX(DATEDIFF(date_published, date_reserved)) as max_days,
    PERCENTILE(DATEDIFF(date_published, date_reserved), 0.5) as median_days
FROM cve_silver.core
WHERE date_reserved IS NOT NULL 
  AND date_published IS NOT NULL
  AND DATEDIFF(date_published, date_reserved) >= 0

avg_days_to_publish,min_days,max_days,median_days
38.8,0,396,16.0


## 3. Risk Distribution - CVSS Score Bucketing

In [0]:
SELECT 
    CASE 
        WHEN cvss_base_score >= 9.0 THEN 'Critical'
        WHEN cvss_base_score >= 7.0 THEN 'High'
        WHEN cvss_base_score >= 4.0 THEN 'Medium'
        WHEN cvss_base_score > 0 THEN 'Low'
        ELSE 'Unknown'
    END as severity,
    COUNT(*) as count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
FROM cve_silver.core
GROUP BY severity
ORDER BY 
    CASE severity 
        WHEN 'Critical' THEN 1 
        WHEN 'High' THEN 2 
        WHEN 'Medium' THEN 3 
        WHEN 'Low' THEN 4 
        ELSE 5 
    END

severity,count,percentage
Critical,1535,4.66
High,6670,20.26
Medium,10144,30.81
Low,876,2.66
Unknown,13699,41.61


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

## 4. CVSS Score Statistics 

In [0]:
SELECT 
    COUNT(*) as total_scored,
    ROUND(AVG(cvss_base_score), 2) as avg_score,
    ROUND(MIN(cvss_base_score), 2) as min_score,
    ROUND(MAX(cvss_base_score), 2) as max_score,
    PERCENTILE(cvss_base_score, 0.5) as median_score
FROM cve_silver.core
WHERE cvss_base_score IS NOT NULL

total_scored,avg_score,min_score,max_score,median_score
19234,6.62,0.0,10.0,6.5


## 5. Top 25 Vendors by Vulnerability Count

In [0]:
SELECT 
    vendor, 
    COUNT(DISTINCT cve_id) as cve_count
FROM cve_silver.affected_products
WHERE vendor IS NOT NULL
GROUP BY vendor
ORDER BY cve_count DESC
LIMIT 25

vendor,cve_count
n/a,5466
Linux,2794
Microsoft,1107
Adobe,741
Unknown,610
SourceCodester,557
Google,546
Apple,468
Oracle Corporation,366
Cisco,278


Databricks visualization. Run in Databricks to view.

## 6. Market Concentration Analysis 

In [0]:
SELECT 
    vendor, 
    COUNT(DISTINCT cve_id) as cve_count
FROM cve_silver.affected_products
WHERE vendor IS NOT NULL
GROUP BY vendor
ORDER BY cve_count DESC
LIMIT 25

vendor,cve_count
n/a,5466
Linux,2794
Microsoft,1107
Adobe,741
Unknown,610
SourceCodester,557
Google,546
Apple,468
Oracle Corporation,366
Cisco,278


## 7. Monthly Trend Analysis (2024)

In [0]:
SELECT 
    MONTH(date_published) as month,
    COUNT(*) as cve_count,
    ROUND(AVG(cvss_base_score), 2) as avg_cvss_score
FROM cve_silver.core
WHERE YEAR(date_published) = 2024 AND cvss_base_score IS NOT NULL
GROUP BY month
ORDER BY month

month,cve_count,avg_cvss_score
1,840,6.3
2,1180,6.37
3,1825,6.56
4,2118,6.44
5,1692,6.58
6,1672,6.54
7,1470,6.71
8,1383,6.75
9,1149,6.82
10,1883,6.77


Databricks visualization. Run in Databricks to view.

## 8. Seasonal Patterns

In [0]:
SELECT 
    CASE 
        WHEN MONTH(date_published) IN (12, 1, 2) THEN 'Winter'
        WHEN MONTH(date_published) IN (3, 4, 5) THEN 'Spring'
        WHEN MONTH(date_published) IN (6, 7, 8) THEN 'Summer'
        ELSE 'Fall'
    END as season,
    COUNT(*) as cve_count,
    ROUND(AVG(cvss_base_score), 2) as avg_severity
FROM cve_silver.core
WHERE cvss_base_score IS NOT NULL
GROUP BY season
ORDER BY 
    CASE season 
        WHEN 'Winter' THEN 1 
        WHEN 'Spring' THEN 2 
        WHEN 'Summer' THEN 3 
        ELSE 4 
    END

season,cve_count,avg_severity
Winter,3744,6.51
Spring,5635,6.52
Summer,4525,6.66
Fall,5330,6.78


Databricks visualization. Run in Databricks to view.

## 9. Unknown/Unscored Vulnerability Identification

In [0]:
SELECT 
    COUNT(*) as total_cves,
    SUM(CASE WHEN cvss_base_score IS NULL THEN 1 ELSE 0 END) as unscored_cves,
    ROUND(SUM(CASE WHEN cvss_base_score IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as unscored_percentage
FROM cve_silver.core

total_cves,unscored_cves,unscored_percentage
32924,13690,41.58


## 10. Vendor-Specific Risk Profiles (Top 10)

In [0]:
SELECT 
    a.vendor,
    COUNT(DISTINCT a.cve_id) as total_cves,
    ROUND(AVG(c.cvss_base_score), 2) as avg_cvss,
    SUM(CASE WHEN c.cvss_base_score >= 9.0 THEN 1 ELSE 0 END) as critical_count,
    SUM(CASE WHEN c.cvss_base_score >= 7.0 AND c.cvss_base_score < 9.0 THEN 1 ELSE 0 END) as high_count
FROM cve_silver.affected_products a
JOIN cve_silver.core c ON a.cve_id = c.cve_id
WHERE a.vendor IS NOT NULL AND c.cvss_base_score IS NOT NULL
GROUP BY a.vendor
ORDER BY total_cves DESC
LIMIT 10

vendor,total_cves,avg_cvss,critical_count,high_count
Microsoft,1104,7.56,266,9511
Adobe,741,6.18,14,232
n/a,558,6.48,46,180
Oracle Corporation,366,5.74,8,82
Cisco,277,6.63,20,107
IBM,263,6.06,10,58
Siemens,247,6.37,114,689
Dell,225,6.43,8,97
Samsung Mobile,220,5.43,0,35
SourceCodester,193,5.46,0,38


Databricks visualization. Run in Databricks to view.

## 11. Summary Statistics

In [0]:
SELECT 
    COUNT(*) as total_cves,
    COUNT(DISTINCT YEAR(date_published)) as years_covered,
    COUNT(cvss_base_score) as scored_cves,
    ROUND(AVG(cvss_base_score), 2) as avg_cvss,
    MIN(date_published) as earliest_date,
    MAX(date_published) as latest_date
FROM cve_silver.core

total_cves,years_covered,scored_cves,avg_cvss,earliest_date,latest_date
32924,1,19234,6.62,2024-01-01T00:00:00.000Z,2024-12-31T23:09:24.244Z
